In [2]:
!pip install skyfield

import pandas as pd
import plotly.express  as px

from skyfield.api import EarthSatellite, load
from datetime import timezone

# =========================================================
# טעינת הנתונים מה-Google Sheets
# =========================================================

url = "https://docs.google.com/spreadsheets/d/1OVwNnElclqJ-Y4tWkePHjQtUetbhLSle/export?format=csv&gid=791204570"

df = pd.read_csv(url)

# =========================================================
# המרת זמן
# =========================================================

df["time"] = pd.to_datetime(
    df["time"],
    format="mixed",
    dayfirst=True,
    utc=True
)

# =========================================================
# סף חריגה
# =========================================================

SEU_THRESHOLD = 0

# רק חריגות
anomalies = df[df["SEU counter"] > SEU_THRESHOLD].copy()

# =========================================================
# TLE של TEVEL2_9
# =========================================================

line1 = "1 63237U 25052AD  26128.22866333  .00012896  00000-0  43681-3 0  9998"
line2 = "2 63237  97.3984  23.6624 0004129 290.4131  69.6666 15.30699981 63911"

satellite = EarthSatellite(line1, line2, "TEVEL2_9")

# =========================================================
# Timescale
# =========================================================

ts = load.timescale()

# =========================================================
# חישוב מיקום עבור כל timestamp
# =========================================================

latitudes = []
longitudes = []
altitudes = []

for _, row in anomalies.iterrows():

    dt = row["time"].to_pydatetime()

    # Skyfield דורש UTC
    dt = dt.replace(tzinfo=timezone.utc)

    # יצירת זמן Skyfield
    t = ts.from_datetime(dt)

    # חישוב מיקום הלוויין
    geocentric = satellite.at(t)

    # הנקודה מתחת ללוויין
    subpoint = geocentric.subpoint()

    latitudes.append(subpoint.latitude.degrees)
    longitudes.append(subpoint.longitude.degrees)
    altitudes.append(subpoint.elevation.km)

# =========================================================
# הוספת הנתונים לטבלה
# =========================================================

anomalies["latitude"] = latitudes
anomalies["longitude"] = longitudes
anomalies["altitude_km"] = altitudes

# =========================================================
# הדפסה לבדיקה
# =========================================================

print(
    anomalies[
        ["time", "SEU counter", "latitude", "longitude", "altitude_km"]
    ]
)

# =========================================================
# שמירת CSV חדש
# =========================================================

anomalies.to_csv(
    "TEVEL2_9_SEU_LOCATIONS.csv",
    index=False
)

# =========================================================
# יצירת גלובוס
# =========================================================

fig = px.scatter_geo(
    anomalies,
    lat="latitude",
    lon="longitude",

    # גודל לפי SEU
    size="SEU counter",

    # צבע קבוע
    color_discrete_sequence=["red"],

    hover_name="time",

    # מפה פרוסה
    projection="equirectangular",

    title="TEVEL2_1 SEU Anomalies"
)

# עיצוב מפה
fig.update_geos(
    showcountries=True,
    showcoastlines=True,
    showland=True,
    landcolor="rgb(230,230,230)",
    oceancolor="rgb(180,220,255)",
    showocean=True
)

# הסרת legend
fig.update_layout(
    height=700,
    showlegend=False,
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

                           time  SEU counter   latitude   longitude  \
8180  2026-03-24 06:28:13+00:00   1774511153  -5.270104 -120.598353   
11809 2026-01-08 06:25:00+00:00   1767904874 -82.160779  170.590690   

       altitude_km  
8180    481.264707  
11809   510.999624  
